In [2]:
import os
import glob
from ruamel.yaml import YAML

def atualizar_sequencias():
    # Inicializa o parser YAML configurado para preservar os comentários
    yaml = YAML()
    yaml.preserve_quotes = True
    
    # Caminho onde os arquivos YAML estão salvos
    diretorio_params = '/home/aki/Desktop/GitHub/Python-VO/params/'
    
    # Dicionário que mapeia o início do nome do arquivo para o valor de 'sequence' desejado.
    # Altere os valores abaixo conforme os nomes reais das suas sequências.
    mapa_sequencias = {
        'cusco': 'cusco_dataset_2',
        'kaist': 'urban27', 
        'kitti': '00'        
    }
    
    # Busca todos os arquivos .yaml no diretório
    arquivos_yaml = glob.glob(os.path.join(diretorio_params, '*.yaml'))
    
    if not arquivos_yaml:
        print(f"Nenhum arquivo YAML encontrado em: {diretorio_params}")
        return

    for caminho_arquivo in arquivos_yaml:
        nome_arquivo = os.path.basename(caminho_arquivo)
        
        # Pega a primeira palavra do nome do arquivo (antes do primeiro '_')
        # Ex: de 'cusco_orb_brutematch.yaml' extrai 'cusco'
        prefixo_dataset = nome_arquivo.split('_')[0].lower()
        
        if prefixo_dataset in mapa_sequencias:
            nova_sequencia = mapa_sequencias[prefixo_dataset]
            
            # 1. Abre e lê o conteúdo atual preservando a formatação
            with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                dados = yaml.load(f)
            
            # 2. Modifica a chave específica, se ela existir
            try:
                sequencia_antiga = dados['dataset']['sequence']
                if sequencia_antiga != nova_sequencia:
                    dados['dataset']['sequence'] = nova_sequencia
                    print(f"[ATUALIZADO] {nome_arquivo} | sequence: {sequencia_antiga} -> {nova_sequencia}")
                else:
                    print(f"[IGNORADO]   {nome_arquivo} | sequence já é '{nova_sequencia}'")
            except KeyError:
                print(f"[ERRO]       {nome_arquivo} | Não possui a chave dataset/sequence.")
                continue
                
            # 3. Salva as alterações de volta no arquivo
            with open(caminho_arquivo, 'w', encoding='utf-8') as f:
                yaml.dump(dados, f)
        else:
            print(f"[AVISO]      {nome_arquivo} | Dataset '{prefixo_dataset}' não reconhecido no mapa.")

if __name__ == '__main__':
    atualizar_sequencias()

[AVISO]      tumrgb_superpoint_supergluematch.yaml | Dataset 'tumrgb' não reconhecido no mapa.
[AVISO]      tumrgb_xfeat_lightergluematch.yaml | Dataset 'tumrgb' não reconhecido no mapa.
[AVISO]      tumrgb_xfeat_flannmatch.yaml | Dataset 'tumrgb' não reconhecido no mapa.
[IGNORADO]   kaist_orb_brutematch.yaml | sequence já é 'urban27'
[IGNORADO]   kitti_orb_brutematch.yaml | sequence já é '00'
[IGNORADO]   kitti_sift_flannmatch.yaml | sequence já é '00'
[AVISO]      robot_orb_bruteforce.yaml | Dataset 'robot' não reconhecido no mapa.
[AVISO]      robot_sift_flannmatch.yaml | Dataset 'robot' não reconhecido no mapa.
[AVISO]      robot_xfeat_bruteforce.yaml | Dataset 'robot' não reconhecido no mapa.
[ATUALIZADO] cusco_sift_flannmatch.yaml | sequence: cusco_dataset_1 -> cusco_dataset_2
[IGNORADO]   kaist_sift_flannmatch.yaml | sequence já é 'urban27'
[ATUALIZADO] cusco_orb_brutematch.yaml | sequence: cusco_dataset_1 -> cusco_dataset_2
[ATUALIZADO] cusco_superpoint_flannmatch.yaml | seque

In [ ]:
import os
import glob
from ruamel.yaml import YAML

def atualizar_configuracoes():
    # Inicializa o parser YAML configurado para preservar os comentários
    yaml = YAML()
    yaml.preserve_quotes = True
    
    # Caminho onde os arquivos YAML estão salvos
    diretorio_params = '/home/aki/Desktop/GitHub/Python-VO/params/'
    
    # 1. MAPEAMENTO DAS SEQUÊNCIAS (Altere conforme sua necessidade)
    mapa_sequencias = {
        'cusco': 'cusco_dataset_1',
        'kaist': 'urban27', 
        'kitti': '00'        
    }
    
    # 2. DEFINIÇÃO DO NÚMERO DE KEYPOINTS DESEJADO
    # Mude este número para o valor que você quer aplicar a todos os arquivos
    novo_numero_keypoints = 1500 
    
    # Busca todos os arquivos .yaml no diretório
    arquivos_yaml = glob.glob(os.path.join(diretorio_params, '*.yaml'))
    
    if not arquivos_yaml:
        print(f"Nenhum arquivo YAML encontrado em: {diretorio_params}")
        return

    for caminho_arquivo in arquivos_yaml:
        nome_arquivo = os.path.basename(caminho_arquivo)
        prefixo_dataset = nome_arquivo.split('_')[0].lower()
        
        if prefixo_dataset in mapa_sequencias:
            nova_sequencia = mapa_sequencias[prefixo_dataset]
            
            # Abre e lê o conteúdo atual preservando a formatação
            with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                dados = yaml.load(f)
            
            alterou_algo = False
            
            # --- AJUSTE DA SEQUÊNCIA ---
            try:
                sequencia_antiga = dados['dataset']['sequence']
                if sequencia_antiga != nova_sequencia:
                    dados['dataset']['sequence'] = nova_sequencia
                    print(f"[{nome_arquivo}] Seq: {sequencia_antiga} -> {nova_sequencia}")
                    alterou_algo = True
            except KeyError:
                print(f"[ERRO] {nome_arquivo} | Não possui a estrutura 'dataset' -> 'sequence'.")
                continue
            
            # --- AJUSTE DINÂMICO DOS KEYPOINTS (nfeatures) ---
            try:
                # Pega o tipo do detector (ex: 'ORB' ou 'SIFT')
                tipo_detector = dados['detector']['type']
                
                # Acessa a subchave correspondente ao tipo do detector de forma dinâmica
                # Ex: dados['detector']['ORB']['nfeatures']
                if tipo_detector in dados['detector'] and 'nfeatures' in dados['detector'][tipo_detector]:
                    keypoints_antigo = dados['detector'][tipo_detector]['nfeatures']
                    
                    if keypoints_antigo != novo_numero_keypoints:
                        dados['detector'][tipo_detector]['nfeatures'] = novo_numero_keypoints
                        print(f"[{nome_arquivo}] Keypoints ({tipo_detector}): {keypoints_antigo} -> {novo_numero_keypoints}")
                        alterou_algo = True
                else:
                    print(f"[AVISO] {nome_arquivo} | Chave 'nfeatures' não encontrada dentro de detector -> {tipo_detector}")
            except KeyError:
                print(f"[AVISO] {nome_arquivo} | Não possui a estrutura padrão de 'detector' -> 'type'.")

            # --- SALVA O ARQUIVO APENAS SE HOUVE MODIFICAÇÃO ---
            if alterou_algo:
                with open(caminho_arquivo, 'w', encoding='utf-8') as f:
                    yaml.dump(dados, f)
                print(f"-> Arquivo {nome_arquivo} salvo com sucesso!\n")
            else:
                print(f"-> Arquivo {nome_arquivo} já estava atualizado.\n")
                
        else:
            print(f"[IGNORADO] {nome_arquivo} | Dataset '{prefixo_dataset}' não está no mapeamento.\n")

if __name__ == '__main__':
    atualizar_configuracoes()